explore and the data

In [1]:
import pandas as pd
df = pd.read_csv("/content/sample_data/job_dataset.csv")
print(df.head())
print(df.info())
print(df.isnull().sum())

   job_id                                      title            location  \
0       1                           Marketing Intern    US, NY, New York   
1       2  Customer Service - Cloud Video Production      NZ, , Auckland   
2       3    Commissioning Machinery Assistant (CMA)       US, IA, Wever   
3       4          Account Executive - Washington DC  US, DC, Washington   
4       5                        Bill Review Manager  US, FL, Fort Worth   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   
2        NaN          NaN  Valor Services provides Workforce Solutions th...   
3      Sales          NaN  Our passion for improving quality of life thro...   
4        NaN          NaN  SpotSource Solutions LLC is a Global Human Cap...   

                                         description  \
0  Foo

In [2]:
df['title'] = df['title'].fillna('')
df['description'] = df['description']. fillna('')
df['requirements'] = df['requirements'].fillna('')
df['company_profile'] = df['company_profile'].fillna('')

In [3]:
df['text'] = (
    df['title'] + " " +
    df['description'] + " " +
    df['requirements'] + " " +
    df['company_profile']
)

In [4]:
df = df[['text', 'fraudulent']]

In [5]:
df.head()
df['fraudulent'].value_counts()

,count
fraudulent,
0,17014
1,866


Preprocessing using TF-IDF

In [6]:
df = df[['text', 'fraudulent']]

df['text'] = df['text'].fillna('')
df['text'] = df['text'].astype(str)

In [7]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [8]:
df['text'] = df['text'].apply(clean_text)

In [9]:
df = df[df['text'].str.strip() != ""]
df = df.reset_index(drop=True)

In [10]:
print(df['text'].iloc[0])
print(df.shape)

marketing intern food a fast growing james beard award winning online food community and crowd sourced and curated recipe hub is currently interviewing full and part time unpaid interns to work in a small team of editors executives and developers in its new york city headquarters reproducing and or repackaging existing food content for a number of partner sites such as huffington post yahoo buzzfeed and more in their various content management systemsresearching blogs and websites for the provisions by food affiliate programassisting in day to day affiliate program support such as screening affiliates and assisting in any affiliate inquiriessupporting with pr amp events when neededhelping with office administrative work such as filing mailing and preparing for meetingsworking with developers to document bugs and suggest improvements to the sitesupporting the marketing and executive staff experience with content management systems a major plus any blogging counts familiar with the food 

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b'
)

X = vectorizer.fit_transform(df['text'])
y = df['fraudulent']

In [12]:
print(X.shape)

(17880, 5000)


Model Tranning

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [14]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

RandomForestClassifier()

In [16]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

MultinomialNB()

In [17]:
y_pred_lr = lr.predict(X_test)
y_pred_nb = nb.predict(X_test)
y_pred_rf = rf.predict(X_test)

In [18]:
from sklearn.metrics import accuracy_score

print("LR Accuracy:", accuracy_score(y_test, y_pred_lr))
print("NB Accuracy:", accuracy_score(y_test, y_pred_nb))
print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))

LR Accuracy: 0.9706375838926175
NB Accuracy: 0.9639261744966443
RF Accuracy: 0.9807046979865772


Model evalution

In [19]:
from sklearn.metrics import classification_report

print("Logistic Regression:\n", classification_report(y_test, y_pred_lr))
print("Naive Bayes:\n", classification_report(y_test, y_pred_nb))
print("Random Forest:\n", classification_report(y_test, y_pred_rf))

Logistic Regression:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98      3395
           1       0.99      0.43      0.59       181

    accuracy                           0.97      3576
   macro avg       0.98      0.71      0.79      3576
weighted avg       0.97      0.97      0.97      3576

Naive Bayes:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      3395
           1       1.00      0.29      0.45       181

    accuracy                           0.96      3576
   macro avg       0.98      0.64      0.71      3576
weighted avg       0.97      0.96      0.95      3576

Random Forest:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99      3395
           1       1.00      0.62      0.76       181

    accuracy                           0.98      3576
   macro avg       0.99      0.81      0.88      3576
weighted avg       0.9

In [20]:
from sklearn.metrics import confusion_matrix

print("LR:\n", confusion_matrix(y_test, y_pred_lr))
print("NB:\n", confusion_matrix(y_test, y_pred_nb))
print("RF:\n", confusion_matrix(y_test, y_pred_rf))

LR:
 [[3394    1]
 [ 104   77]]
NB:
 [[3395    0]
 [ 129   52]]
RF:
 [[3395    0]
 [  69  112]]


Prediction system

In [21]:
def predict_job(text, threshold=0.4):
    text = clean_text(text)

    text_vector = vectorizer.transform([text])

    prob_fake = rf.predict_proba(text_vector)[0][1]

    if prob_fake > threshold:
        return f"Fake job "
    else:
        return f"Real Job "

some inputs

In [22]:
sample = "Earn $5000 per week from home. No experience required. Limited seats available. Apply now!"
print(predict_job(sample))

Fake job 


In [23]:
s2 = "No interview, no skills needed. Get paid instantly after joining."
print(predict_job(s2, threshold=0.2))

Fake job 


In [24]:
s3 = "Seeking a UI/UX designer with experience in Figma, user research, and prototyping."
print(predict_job(s3))

Real Job 


In [25]:
s4 = "The candidate must have a bachelor's degree in commerce and knowledge of accounting software."
print(predict_job(s4))

Real Job 


In [26]:
s5 = "Guaranteed income opportunity. Work from anywhere and earn fast money."
print(predict_job(s5, threshold=0.3))

Fake job 
